# Logistic Regression con validacion LOSO (etiquetas individuales)

Este notebook implementa el **Experimento A** con features derivadas de HR/R-R y ECG.

El objetivo es evaluar la generalizacion a un trabajador no visto mediante Leave-One-Subject-Out Cross-Validation (LOSO). Cada trabajador obtiene sus propios umbrales P33/P66 calculados sobre todo su registro `FatigueIndex`, y las etiquetas resultantes se guardan en `FatigueLevel`.

## Etiquetado individual

1. Para cada trabajador se calculan P33 y P66 usando todo su `FatigueIndex` valido.
2. Se crean las etiquetas `Low`, `Medium` y `High` en `FatigueLevel` con esos umbrales propios.
3. LOSO usa directamente `FatigueLevel` en TRAIN y TEST; no se recalculan umbrales globales dentro del fold.
4. Los imputadores y scalers se ajustan solo con TRAIN.

Las entradas son las columnas `_z`, normalizadas respecto a la baseline individual de cada trabajador. `FatigueIndex` no se usa como feature.

In [10]:
from pathlib import Path

import json
import joblib
import numpy as np
import onnx
import onnxruntime as ort
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

current_dir = Path.cwd()
candidate_roots = [
    current_dir,
    current_dir.parent,
    current_dir.parent.parent,
    current_dir.parent.parent.parent,
    current_dir / 'new',
]
ROOT_DIR = next(
    (
        candidate
        for candidate in candidate_roots
        if (candidate / 'W00' / 'PROCESSED' / 'combined_features_1min.csv').exists()
    ),
    current_dir,
)
DATA_DIR = ROOT_DIR
BASELINE_WINDOW_COUNT = 15
TARGET_COLUMN_CANDIDATES = ['FatigueIndex', 'fatigue_index']
LABELS = ['Low', 'Medium', 'High']
C_GRID = [0.01, 0.1, 1, 10, 100.0, 300.0, 1000.0, 3000.0, 10000.0]

print(f'Directorio de datos: {DATA_DIR}')

Directorio de datos: c:\Users\Carlo\Desktop\owncloud 2025-08-11 ECG\new


## Carga de datos

Cada CSV corresponde a un trabajador. El nombre de la carpeta (`W00`, ..., `W08`) se conserva como identificador para construir los folds, pero nunca entra como feature del modelo.

In [12]:
def make_labels(fatigue_values, p33, p66):
    return pd.cut(
        fatigue_values,
        bins=[-np.inf, p33, p66, np.inf],
        labels=LABELS,
        right=False,
    ).astype(object)


worker_files = {
    worker_dir.name: worker_dir / 'PROCESSED' / 'combined_features_1min.csv'
    for worker_dir in sorted(DATA_DIR.glob('W[0-9][0-9]'))
    if (worker_dir / 'PROCESSED' / 'combined_features_1min.csv').exists()
}
if len(worker_files) < 2:
    raise ValueError(f'Se necesitan al menos 2 trabajadores procesados y se encontraron {len(worker_files)}: {list(worker_files)}')

data = {}
for worker, path in worker_files.items():
    frame = pd.read_csv(path)
    target_matches = [column for column in TARGET_COLUMN_CANDIDATES if column in frame.columns]
    if len(target_matches) != 1:
        raise ValueError(f'{worker}: no se encontro exactamente una columna FatigueIndex/fatigue_index')
    frame = frame.rename(columns={target_matches[0]: 'FatigueIndex'})
    worker_fatigue = pd.to_numeric(frame['FatigueIndex'], errors='coerce')
    valid_fatigue = worker_fatigue.dropna()
    if valid_fatigue.empty:
        raise ValueError(f'{worker}: no contiene valores validos de FatigueIndex')
    worker_p33, worker_p66 = np.percentile(valid_fatigue, [33, 66])
    frame['P33_worker'] = worker_p33
    frame['P66_worker'] = worker_p66
    frame['FatigueLevel'] = make_labels(worker_fatigue, worker_p33, worker_p66)
    frame['Trabajador'] = worker
    data[worker] = frame

feature_columns = [
    'ECG_energy_z', 'ECG_mean_z', 'ECG_missing_peaks_z', 'ECG_range_z',
    'ECG_samp_ent_z', 'ECG_std_z', 'HR_max_z', 'HR_mean_z', 'HR_min_z',
    'RMSSD_z', 'SDNN_z', 'pNN50_z', 'HR_mean_ultimos_5min_z',
    'cambio_HR_vs_baseline_z', 'tendencia_HR_z', 'cambio_RMSSD_z',
]
missing_features = [column for column in feature_columns if column not in data[next(iter(data))].columns]
if missing_features:
    raise ValueError(f'Faltan features requeridas: {missing_features}')
if not feature_columns:
    raise ValueError('No se encontraron columnas de entrada terminadas en _z')

print('Trabajadores:', list(data))
print('Features del Experimento A:', feature_columns)
print('Ventanas:', {worker: len(frame) for worker, frame in data.items()})

Trabajadores: ['W00', 'W01', 'W02', 'W03', 'W04', 'W05', 'W06', 'W07', 'W08', 'W09']
Features del Experimento A: ['ECG_energy_z', 'ECG_mean_z', 'ECG_missing_peaks_z', 'ECG_range_z', 'ECG_samp_ent_z', 'ECG_std_z', 'HR_max_z', 'HR_mean_z', 'HR_min_z', 'RMSSD_z', 'SDNN_z', 'pNN50_z', 'HR_mean_ultimos_5min_z', 'cambio_HR_vs_baseline_z', 'tendencia_HR_z', 'cambio_RMSSD_z']
Ventanas: {'W00': 104, 'W01': 637, 'W02': 603, 'W03': 407, 'W04': 562, 'W05': 74, 'W06': 35, 'W07': 67, 'W08': 133, 'W09': 16}


## Funciones de target y evaluacion

Los percentiles se calculan en cada fold y solo con el vector de `FatigueIndex` de TRAIN. Los valores faltantes del target no se usan para calcular percentiles ni para entrenar/evaluar ese fold.

In [ ]:
def make_labels(fatigue_values, p33, p66):
    return pd.cut(fatigue_values, bins=[-np.inf, p33, p66, np.inf], labels=LABELS, right=False).astype(object)


def build_model(C=1.0):
    return Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('classifier', LogisticRegression(
            C=C, penalty='l2', class_weight='balanced', max_iter=2000, random_state=42,
        )),
    ])


def optimize_model(X_train, y_train):
    inner_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    search = GridSearchCV(
        estimator=build_model(), param_grid={'classifier__C': C_GRID},
        scoring='balanced_accuracy', cv=inner_cv, refit=True, n_jobs=-1,
    )
    search.fit(X_train, y_train)
    return search

## Experimento A

Features utilizadas: todas las columnas `_z` disponibles, que corresponden a features derivadas de HR/R-R y ECG.

La matriz de confusion utiliza siempre el orden `Low`, `Medium`, `High`.

In [ ]:
fold_results = []
confusion_matrices = {}

for test_worker in sorted(data):
    train_workers = [worker for worker in sorted(data) if worker != test_worker]
    train_frame = pd.concat([data[worker] for worker in train_workers], ignore_index=True)
    test_frame = data[test_worker].copy()
    train_target = train_frame['FatigueLevel']
    test_target = test_frame['FatigueLevel']
    train_mask = train_target.notna()
    test_mask = test_target.notna()
    X_train = train_frame.loc[train_mask, feature_columns]
    y_train = train_target.loc[train_mask]
    X_test = test_frame.loc[test_mask, feature_columns]
    y_test = test_target.loc[test_mask]
    if y_train.nunique() < 2:
        raise ValueError(f'{test_worker}: TRAIN tiene menos de dos clases')
    search = optimize_model(X_train, y_train)
    y_pred = search.predict(X_test)
    confusion_matrices[test_worker] = confusion_matrix(y_test, y_pred, labels=LABELS)
    matrix = confusion_matrices[test_worker]
    class_recall = np.divide(np.diag(matrix), matrix.sum(axis=1), out=np.zeros(3, dtype=float), where=matrix.sum(axis=1) != 0)
    fold_results.append({
        'Test_worker': test_worker, 'Train_workers': ', '.join(train_workers),
        'P33_worker_test': test_frame['P33_worker'].iloc[0], 'P66_worker_test': test_frame['P66_worker'].iloc[0],
        'N_train': len(y_train), 'N_test': len(y_test), 'Best_C': search.best_params_['classifier__C'],
        'Recall_Low': class_recall[0], 'Recall_Medium': class_recall[1], 'Recall_High': class_recall[2],
        'Accuracy': accuracy_score(y_test, y_pred),
        'Balanced_Accuracy': balanced_accuracy_score(y_test, y_pred),
        'Macro_F1': f1_score(y_test, y_pred, labels=LABELS, average='macro', zero_division=0),
    })

fold_results_df = pd.DataFrame(fold_results)
mean_row = {column: np.nan for column in fold_results_df.columns}
mean_row['Test_worker'] = 'Media'
mean_row['Train_workers'] = 'Promedio de los folds'
for metric in ['P33_worker_test', 'P66_worker_test', 'N_train', 'N_test', 'Best_C', 'Recall_Low', 'Recall_Medium', 'Recall_High', 'Accuracy', 'Balanced_Accuracy', 'Macro_F1']:
    mean_row[metric] = fold_results_df[metric].mean()
fold_results_with_mean = pd.concat([fold_results_df, pd.DataFrame([mean_row])], ignore_index=True)
fold_results_with_mean

In [ ]:
print('Matrices de confusion por trabajador TEST:')
for worker, matrix in confusion_matrices.items():
    print(f'\nTEST = {worker}')
    print(pd.DataFrame(matrix, index=LABELS, columns=LABELS))

metrics = ['Accuracy', 'Balanced_Accuracy', 'Macro_F1']
summary_df = pd.DataFrame({'Metric': metrics, 'Mean': [fold_results_df[metric].mean() for metric in metrics], 'Std': [fold_results_df[metric].std(ddof=1) for metric in metrics]})
print('Media y desviacion estandar de las metricas:')
display(summary_df)

aggregate_confusion = np.sum(np.stack([confusion_matrices[worker] for worker in sorted(confusion_matrices)]), axis=0)
display(pd.DataFrame(aggregate_confusion, index=LABELS, columns=LABELS))
class_support = aggregate_confusion.sum(axis=1)
predicted_support = aggregate_confusion.sum(axis=0)
class_metrics_df = pd.DataFrame({'Support': class_support, 'Recall': np.divide(np.diag(aggregate_confusion), class_support, out=np.zeros(3, dtype=float), where=class_support != 0), 'Precision': np.divide(np.diag(aggregate_confusion), predicted_support, out=np.zeros(3, dtype=float), where=predicted_support != 0)}, index=LABELS)
display(class_metrics_df.round(3))

distribution_rows = []
for worker, frame in data.items():
    counts = frame['FatigueLevel'].value_counts().reindex(LABELS, fill_value=0)
    distribution_rows.append({'Test_worker': worker, 'Low': int(counts['Low']), 'Medium': int(counts['Medium']), 'High': int(counts['High']), 'Total_valid': int(counts.sum())})
print('Distribucion individual de clases por trabajador:')
display(pd.DataFrame(distribution_rows))

In [ ]:
comparison_df = pd.DataFrame({
    'Modelo': [
        'Original sin variables temporales',
        'Temporal original',
        'Optimizado anterior',
        'Optimizado con cuadrícula ampliada',
        'Optimizado con cuadrícula ampliada',
    ],
    'class_weight': ['None', 'None', 'balanced', 'balanced', 'balanced'],
    'C por fold': [
        '1.0',
        '1.0',
        '100 en los 5 folds',
        'W00: 10000; W01: 3000; W02: 10000; W03: 3000; ...',
        'W00: 1000; W01: 10000; W02: 3000; W03: 3000; W...',
    ],
    'C final': [
        '1.0',
        '1.0',
        '100',
        '10000.0',
        '3000.0',
    ],
    'Accuracy': [
        0.781303,
        0.783572,
        0.844313,
        0.8510,
        0.781666,
    ],
    'Balanced Accuracy': [
        0.679752,
        0.692432,
        0.702434,
        0.7319,
        0.742559,
    ],
    'Macro-F1': [
        0.652589,
        0.667675,
        0.681970,
        0.7111,
        0.700735,
    ],
})

for column in ['Accuracy', 'Balanced Accuracy', 'Macro-F1']:
    comparison_df[column] = comparison_df[column].map(
        lambda value: f'{value:.2%}'
    )

display(comparison_df)

## Modelo final y exportacion

La evaluacion LOSO anterior mide la generalizacion dejando un trabajador fuera en cada fold. Despues, este bloque reentrena un modelo final con todos los trabajadores y todas las ventanas validas. Este modelo final es el que se utilizara para la aplicacion.

El modelo y el preprocesamiento completo se guardan en formato `joblib`, el modelo para Expo/React Native en formato `ONNX`, y la configuracion en un archivo `JSON`. Los modelos individuales se guardan en `new/models/individual/LOGISTIC_REGRESSION/`, separados de los modelos globales.

In [ ]:
# Reentrenamiento final con etiquetas relativas a cada trabajador
all_frame = pd.concat([data[worker] for worker in sorted(data)], ignore_index=True)
all_target = all_frame['FatigueLevel']
all_mask = all_target.notna()
X_all = all_frame.loc[all_mask, feature_columns]
y_all = all_target.loc[all_mask]

final_search = optimize_model(X_all, y_all)
final_model = final_search.best_estimator_

models_dir = ROOT_DIR / 'models' / 'individual' / 'LOGISTIC_REGRESSION'
models_dir.mkdir(parents=True, exist_ok=True)
joblib_path = models_dir / 'final_logistic_regression_individual.joblib'
onnx_path = models_dir / 'final_logistic_regression_individual.onnx'
metadata_path = models_dir / 'final_logistic_regression_individual_metadata.json'

joblib.dump(final_model, joblib_path)
onnx_model = convert_sklearn(final_model, initial_types=[('features', FloatTensorType([None, len(feature_columns)]))])
onnx_path.write_bytes(onnx_model.SerializeToString())

metadata = {
    'model_type': 'LogisticRegression', 'validation': 'LOSO_individual_worker_percentiles_with_inner_C_search',
    'training_workers': sorted(data), 'feature_columns': feature_columns, 'feature_count': len(feature_columns),
    'labels': LABELS, 'label_definition': 'P33/P66 calculated independently for each worker over the full FatigueIndex record',
    'class_weight': 'balanced', 'C_grid': C_GRID,
    'selected_C_final': float(final_search.best_params_['classifier__C']),
    'target_column': 'FatigueIndex', 'target_label_column': 'FatigueLevel',
}
metadata_path.write_text(json.dumps(metadata, indent=2), encoding='utf-8')
print(f'Modelo final entrenado con {len(y_all)} ventanas y {len(feature_columns)} features')
print(f'C final seleccionado: {final_search.best_params_["classifier__C"]}')
print(f'Guardado en: {models_dir}')

## Experimento B: ablacion HR/R-R

Se conservan solo las features calculadas a partir de HR/R-R y las cuatro features temporales. Se eliminan las features ECG. El target sigue siendo el `FatigueIndex` categorico creado dentro de cada fold con P33 y P66 de TRAIN.

In [ ]:
HR_RR_FEATURES = ['SDNN_z', 'RMSSD_z', 'pNN50_z', 'HR_mean_z', 'HR_max_z', 'HR_min_z', 'HR_mean_ultimos_5min_z', 'cambio_HR_vs_baseline_z', 'tendencia_HR_z', 'cambio_RMSSD_z']
ECG_FEATURES = ['ECG_energy_z', 'ECG_mean_z', 'ECG_missing_peaks_z', 'ECG_range_z', 'ECG_samp_ent_z', 'ECG_std_z']


def run_ablation_experiment(experiment_name, selected_features):
    results = []
    for test_worker in sorted(data):
        train_workers = [worker for worker in sorted(data) if worker != test_worker]
        train_frame = pd.concat([data[worker] for worker in train_workers], ignore_index=True)
        test_frame = data[test_worker].copy()
        train_target, test_target = train_frame['FatigueLevel'], test_frame['FatigueLevel']
        train_mask, test_mask = train_target.notna(), test_target.notna()
        X_train, y_train = train_frame.loc[train_mask, selected_features], train_target.loc[train_mask]
        X_test, y_test = test_frame.loc[test_mask, selected_features], test_target.loc[test_mask]
        search = optimize_model(X_train, y_train)
        y_pred = search.predict(X_test)
        results.append({'Experiment': experiment_name, 'Test_worker': test_worker, 'N_train': len(y_train), 'N_test': len(y_test), 'Feature_count': len(selected_features), 'Best_C': search.best_params_['classifier__C'], 'Accuracy': accuracy_score(y_test, y_pred), 'Balanced_Accuracy': balanced_accuracy_score(y_test, y_pred), 'Macro_F1': f1_score(y_test, y_pred, labels=LABELS, average='macro', zero_division=0)})
    return pd.DataFrame(results)


def add_mean_row(results_frame):
    mean_row = {column: np.nan for column in results_frame.columns}
    mean_row['Test_worker'] = 'Media'
    for column in results_frame.select_dtypes(include='number').columns:
        mean_row[column] = results_frame[column].mean()
    return pd.concat([results_frame, pd.DataFrame([mean_row])], ignore_index=True)


hr_rr_results = run_ablation_experiment('Solo HR/RR + temporales', HR_RR_FEATURES)
hr_rr_results_with_mean = add_mean_row(hr_rr_results)
print('Resultados del Experimento B por trabajador y promedio:')
display(hr_rr_results_with_mean.round(4))

## Experimento C: ablacion ECG

Se conservan solo las features calculadas a partir del ECG. Se excluyen las features HR/R-R y las temporales, porque estas ultimas dependen de HR/R-R. `ECG_missing_peaks` se conserva como feature de calidad de picos ECG. El target sigue siendo el `FatigueIndex` categorico creado dentro de cada fold con P33 y P66 de TRAIN.

In [ ]:
ecg_results = run_ablation_experiment('Solo ECG', ECG_FEATURES)
ecg_results_with_mean = add_mean_row(ecg_results)
print('Resultados del Experimento C por trabajador y promedio:')
display(ecg_results_with_mean.round(4))

ec_results = pd.concat([hr_rr_results, ecg_results], ignore_index=True)
ablation_summary = (
    ec_results.groupby('Experiment')[['Accuracy', 'Balanced_Accuracy', 'Macro_F1']]
    .agg(['mean', 'std'])
    .round(4)
)
print('Resumen de las ablaciones:')
display(ablation_summary)

In [ ]:
import matplotlib.pyplot as plt


def fit_final_feature_model(selected_features):
    X_selected = all_frame.loc[all_mask, selected_features]
    return optimize_model(X_selected, y_all).best_estimator_


models_by_experiment = {
    'A: ECG + HR/RR': (feature_columns, fit_final_feature_model(feature_columns)),
    'B: HR/RR + temporales': (HR_RR_FEATURES, fit_final_feature_model(HR_RR_FEATURES)),
    'C: ECG': (ECG_FEATURES, fit_final_feature_model(ECG_FEATURES)),
}

importance_tables = {}
for experiment_name, (selected_features, model) in models_by_experiment.items():
    coefficients = model.named_steps['classifier'].coef_
    importance = np.mean(np.abs(coefficients), axis=0)
    importance_tables[experiment_name] = (
        pd.DataFrame({'Feature': selected_features, 'Importance': importance})
        .sort_values('Importance', ascending=True)
    )

fig, axes = plt.subplots(1, 3, figsize=(18, 7), constrained_layout=True)
for axis, (experiment_name, importance_table) in zip(axes, importance_tables.items()):
    axis.barh(importance_table['Feature'], importance_table['Importance'], color='#2f6f8f')
    axis.set_title(experiment_name)
    axis.set_xlabel('Importancia media: |coeficiente|')
    axis.grid(axis='x', alpha=0.25)

fig.suptitle('Feature importance de los experimentos A, B y C')
plt.show()

print('Features ordenadas por importancia:')
for experiment_name, importance_table in importance_tables.items():
    print(f'\n{experiment_name}')
    display(importance_table.sort_values('Importance', ascending=False).round(4))